# AI Research Paper Intelligence System
## Notebook 2: Search | BART | KeyBERT | TF-IDF | Groq LLaMA | Graphs | Compare | PDF | Bookmarks
**CBSOT Summer Internship 2026**

Run Notebook 1 first to build the FAISS index!

In [ ]:
!pip install datasets sentence-transformers faiss-cpu transformers keybert torch networkx reportlab scikit-learn matplotlib seaborn groq -q
print('All dependencies installed!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
import json
import torch
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from keybert import KeyBERT
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from datetime import datetime
from groq import Groq

os.makedirs('outputs', exist_ok=True)
print('Libraries imported!')

In [ ]:
index      = faiss.read_index('outputs/faiss_index/papers.index')
df         = pd.read_pickle('outputs/data/papers_df.pkl')
embeddings = np.load('outputs/data/embeddings.npy')
model      = SentenceTransformer('all-MiniLM-L6-v2')

print(f'Index      : {index.ntotal:,} papers')
print(f'DataFrame  : {df.shape}')
print(f'Embeddings : {embeddings.shape}')

## Semantic Search with MMR

In [ ]:
def semantic_search(query, top_k=10, mmr=True, diversity=0.3):
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    fetch_k = top_k * 3 if mmr else top_k
    scores, indices = index.search(q_emb, fetch_k)
    results = [{'idx': int(idx), 'title': df['title'].iloc[idx],
                'abstract': df['abstract'].iloc[idx], 'score': float(score)}
               for idx, score in zip(indices[0], scores[0])]
    if not mmr:
        return results[:top_k]
    candidate_embs = np.array([embeddings[r['idx']] for r in results])
    selected = [0]
    while len(selected) < top_k:
        remaining = [i for i in range(len(results)) if i not in selected]
        if not remaining:
            break
        mmr_scores = []
        for i in remaining:
            rel = results[i]['score']
            sim = max(cosine_similarity(candidate_embs[i].reshape(1,-1),
                                       candidate_embs[j].reshape(1,-1))[0][0]
                      for j in selected)
            mmr_scores.append((1-diversity)*rel - diversity*sim)
        selected.append(remaining[np.argmax(mmr_scores)])
    return [results[i] for i in selected]

results = semantic_search('graph neural networks', top_k=5)
for i, r in enumerate(results):
    print(f'{i+1}. [{r["score"]:.4f}] {r["title"]}')

## TF-IDF Matrix Build

In [ ]:
print('Building TF-IDF matrix on 50k papers...')
tfidf_vectorizer = TfidfVectorizer(max_features=10000, stop_words='english',
                                    ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text'].tolist())
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')

def tfidf_search(query, top_k=10):
    q_vec   = tfidf_vectorizer.transform([query])
    scores  = cos_sim(q_vec, tfidf_matrix).flatten()
    top_idx = scores.argsort()[::-1][:top_k]
    return [{'idx': int(i), 'title': df['title'].iloc[i],
             'abstract': df['abstract'].iloc[i],
             'score': float(scores[i])} for i in top_idx]

print('TF-IDF search ready!')

## TF-IDF 1: Semantic vs TF-IDF Search Comparison

In [ ]:
def compare_search_methods(query, top_k=5):
    semantic_results = semantic_search(query, top_k=top_k, mmr=False)
    tfidf_results    = tfidf_search(query, top_k=top_k)
    common = set(r['title'] for r in semantic_results) & set(r['title'] for r in tfidf_results)

    print('='*70)
    print(f'Query: "{query}"')
    print('='*70)
    print(f'\nSEMANTIC SEARCH (all-MiniLM-L6-v2 + FAISS):')
    for i, r in enumerate(semantic_results):
        print(f'  {i+1}. [{r["score"]:.4f}] {r["title"][:65]}')
    print(f'\nTF-IDF SEARCH (sklearn, ngram 1-2):')
    for i, r in enumerate(tfidf_results):
        print(f'  {i+1}. [{r["score"]:.4f}] {r["title"][:65]}')
    print(f'\nCommon results: {len(common)}/{top_k}')

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, res, title, color in [
        (axes[0], semantic_results, 'Semantic Search (MiniLM + FAISS)', '#3498db'),
        (axes[1], tfidf_results,    'TF-IDF Search (sklearn)',           '#e74c3c')
    ]:
        labels = [r['title'][:40]+'...' for r in res]
        scores = [r['score'] for r in res]
        bars   = ax.barh(range(len(labels)), scores, color=color, alpha=0.8)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=8)
        ax.set_title(title, fontweight='bold', fontsize=11)
        ax.set_xlabel('Score')
        for bar, s in zip(bars, scores):
            ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
                    f'{s:.3f}', va='center', fontsize=8)
    plt.suptitle(f'Semantic vs TF-IDF | Query: "{query}"', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/06_semantic_vs_tfidf.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/06_semantic_vs_tfidf.png')

compare_search_methods('attention mechanism transformer', top_k=5)

## BART Summarization

In [ ]:
print('Loading BART summarizer...')
tokenizer  = AutoTokenizer.from_pretrained('sshleifer/distilbart-cnn-12-6')
bart_model = AutoModelForSeq2SeqLM.from_pretrained('sshleifer/distilbart-cnn-12-6')
print('BART loaded!')

def summarize(text, max_len=120, min_len=40):
    try:
        inputs = tokenizer(text[:1024], return_tensors='pt',
                           truncation=True, max_length=1024)
        with torch.no_grad():
            summary_ids = bart_model.generate(
                inputs['input_ids'], max_length=max_len,
                min_length=min_len, num_beams=4, early_stopping=True
            )
        return tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    except:
        return text[:300] + '...'

print(f'Summary: {summarize(results[0]["abstract"])}')

## KeyBERT Keyword Extraction

In [ ]:
kw_model = KeyBERT()

def extract_keywords(text, top_n=8):
    kws = kw_model.extract_keywords(text, keyphrase_ngram_range=(1,2),
                                    stop_words='english', use_mmr=True,
                                    diversity=0.5, top_n=top_n)
    return [k[0] for k in kws]

print(f'Keywords: {extract_keywords(results[0]["abstract"])}')

## TF-IDF 2: KeyBERT vs TF-IDF Keyword Comparison

In [ ]:
def compare_keywords(text, top_n=8):
    keybert_kws = extract_keywords(text, top_n=top_n)

    doc_vec       = tfidf_vectorizer.transform([text])
    feature_names = tfidf_vectorizer.get_feature_names_out()
    tfidf_scores  = doc_vec.toarray().flatten()
    top_idx       = tfidf_scores.argsort()[::-1][:top_n]
    tfidf_kws     = [feature_names[i] for i in top_idx if tfidf_scores[i] > 0]

    common = set(keybert_kws) & set(tfidf_kws)
    print(f'KeyBERT : {keybert_kws}')
    print(f'TF-IDF  : {tfidf_kws}')
    print(f'Common  : {common if common else "None"}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, kws, title, color in [
        (axes[0], keybert_kws, 'KeyBERT Keywords', '#2ecc71'),
        (axes[1], tfidf_kws,   'TF-IDF Keywords',  '#f39c12')
    ]:
        ax.barh(range(len(kws)), [1]*len(kws), color=color, alpha=0.8)
        ax.set_yticks(range(len(kws)))
        ax.set_yticklabels(kws, fontsize=10)
        ax.set_title(title, fontweight='bold')
        ax.set_xticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    plt.suptitle('KeyBERT vs TF-IDF Keyword Extraction', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/07_keybert_vs_tfidf.png', dpi=150, bbox_inches='tight')
    plt.show()
    return keybert_kws, tfidf_kws

compare_keywords(results[0]['abstract'])

## Groq LLaMA Integration

In [ ]:
GROQ_API_KEY = "your_groq_api_key_here"  # get free key from console.groq.com
groq_client  = Groq(api_key=GROQ_API_KEY)

def groq_summarize(text, max_words=100):
    prompt = f"""Summarize this ML research paper abstract in {max_words} words or less.
Be concise. Highlight the key contribution and method.

Abstract: {text[:2000]}

Summary:"""
    try:
        resp = groq_client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=0.3
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"Groq error: {e}"

def groq_search_insights(query, results):
    titles = '\n'.join([f'{i+1}. {r["title"]}' for i, r in enumerate(results)])
    prompt = f"""These are top ML research papers for query: \"{query}\"

Papers:
{titles}

In 3-4 sentences, what is the current research trend in this area based on these papers?"""
    try:
        resp = groq_client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=0.5
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"Groq error: {e}"

def groq_compare_papers(r1, r2, cross_sim):
    prompt = f"""You are a research assistant. Compare these two ML papers briefly.

Paper A: {r1['title']}
Abstract A: {r1['abstract'][:600]}

Paper B: {r2['title']}
Abstract B: {r2['abstract'][:600]}

Semantic Similarity Score: {cross_sim:.4f}

Provide:
1. Key difference in approach (1-2 sentences)
2. Which problem each solves (1 sentence each)
3. Which is more novel and why (1 sentence)
Keep total response under 150 words."""
    try:
        resp = groq_client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300, temperature=0.4
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"Groq error: {e}"

print('Groq client ready!')

# Test
query   = 'attention mechanism transformer'
results = semantic_search(query, top_k=5)
print('\n--- Groq Summarization ---')
print(groq_summarize(results[0]['abstract']))
print('\n--- Groq Search Insights ---')
print(groq_search_insights(query, results))

## Paper Similarity Graph

In [ ]:
def plot_similarity_graph(query, top_k=8, threshold=0.72):
    os.makedirs('outputs', exist_ok=True)
    results    = semantic_search(query, top_k=top_k, mmr=False)
    paper_embs = np.array([embeddings[r['idx']] for r in results])
    sim_matrix = cosine_similarity(paper_embs)

    G = nx.Graph()
    for i, r in enumerate(results):
        short = r['title'][:40]+'...' if len(r['title'])>40 else r['title']
        G.add_node(i, title=short, score=r['score'])
    for i in range(len(results)):
        for j in range(i+1, len(results)):
            if sim_matrix[i][j] > threshold:
                G.add_edge(i, j, weight=float(sim_matrix[i][j]))

    plt.figure(figsize=(14, 9))
    pos         = nx.spring_layout(G, seed=42, k=2)
    scores      = [G.nodes[n]['score'] for n in G.nodes]
    node_colors = plt.cm.YlOrRd(np.array(scores)/max(scores))
    edge_weights= [G[u][v]['weight']*3 for u,v in G.edges] if G.edges else [1]

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2000, alpha=0.9)
    if G.edges:
        nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.5, edge_color='#3498db')
        nx.draw_networkx_edge_labels(G, pos,
            {(u,v): f"{G[u][v]['weight']:.2f}" for u,v in G.edges}, font_size=6)
    nx.draw_networkx_labels(G, pos, {n: G.nodes[n]['title'] for n in G.nodes}, font_size=7)
    plt.title(f'Paper Similarity Graph — "{query}"', fontweight='bold', fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('outputs/03_similarity_graph.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}')

plot_similarity_graph('attention mechanism transformer', top_k=8)

## Citation Network

In [ ]:
def plot_citation_network(queries, papers_per=5):
    os.makedirs('outputs', exist_ok=True)
    colors_list = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
    all_papers  = []
    for ti, query in enumerate(queries):
        for r in semantic_search(query, top_k=papers_per, mmr=True):
            r['topic_idx'] = ti
            all_papers.append(r)

    G = nx.Graph()
    for i, p in enumerate(all_papers):
        short = p['title'][:32]+'...' if len(p['title'])>32 else p['title']
        G.add_node(i, title=short, color=colors_list[p['topic_idx']%len(colors_list)])

    paper_embs = np.array([embeddings[p['idx']] for p in all_papers])
    sim_matrix = cosine_similarity(paper_embs)
    for i in range(len(all_papers)):
        for j in range(i+1, len(all_papers)):
            same = all_papers[i]['topic_idx'] == all_papers[j]['topic_idx']
            th   = 0.70 if same else 0.82
            if sim_matrix[i][j] > th:
                G.add_edge(i, j, weight=float(sim_matrix[i][j]), cross=not same)

    plt.figure(figsize=(16, 11))
    pos         = nx.spring_layout(G, seed=42, k=1.5)
    node_colors = [G.nodes[n]['color'] for n in G.nodes]
    intra = [(u,v) for u,v,d in G.edges(data=True) if not d.get('cross')]
    cross = [(u,v) for u,v,d in G.edges(data=True) if d.get('cross')]

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1400, alpha=0.85)
    if intra:
        nx.draw_networkx_edges(G, pos, edgelist=intra, width=1.5, alpha=0.4, edge_color='gray')
    if cross:
        nx.draw_networkx_edges(G, pos, edgelist=cross, width=2.5, alpha=0.8,
                               edge_color='black', style='dashed')
    nx.draw_networkx_labels(G, pos, {n: G.nodes[n]['title'] for n in G.nodes}, font_size=6)

    from matplotlib.patches import Patch
    legend = [Patch(color=colors_list[i%len(colors_list)], label=q[:28])
              for i, q in enumerate(queries)]
    plt.legend(handles=legend, loc='upper left', fontsize=9, title='Topics')
    plt.title('Citation Network (Dashed = Cross-topic links)', fontweight='bold', fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('outputs/04_citation_network.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()} | Cross: {len(cross)}')

plot_citation_network([
    'transformer attention mechanism',
    'reinforcement learning policy gradient',
    'generative adversarial networks'
], papers_per=5)

## TF-IDF 3: Compare 2 Papers (Semantic + TF-IDF + Groq)

In [ ]:
def compare_papers(query1, query2):
    r1 = semantic_search(query1, top_k=1, mmr=False)[0]
    r2 = semantic_search(query2, top_k=1, mmr=False)[0]

    # Semantic similarity
    emb1      = embeddings[r1['idx']].reshape(1,-1)
    emb2      = embeddings[r2['idx']].reshape(1,-1)
    cross_sim = cosine_similarity(emb1, emb2)[0][0]

    # TF-IDF similarity
    tfidf_emb1 = tfidf_vectorizer.transform([r1['abstract']])
    tfidf_emb2 = tfidf_vectorizer.transform([r2['abstract']])
    tfidf_sim  = cos_sim(tfidf_emb1, tfidf_emb2)[0][0]

    sum1 = summarize(r1['abstract'])
    sum2 = summarize(r2['abstract'])
    kw1  = extract_keywords(r1['abstract'])
    kw2  = extract_keywords(r2['abstract'])
    common = set(kw1) & set(kw2)

    print('='*70)
    print('PAPER COMPARISON')
    print('='*70)
    print(f'\nPAPER A: {r1["title"]}')
    print(f'  Score   : {r1["score"]:.4f}')
    print(f'  Summary : {sum1}')
    print(f'  Keywords: {kw1}')
    print(f'\nPAPER B: {r2["title"]}')
    print(f'  Score   : {r2["score"]:.4f}')
    print(f'  Summary : {sum2}')
    print(f'  Keywords: {kw2}')
    print(f'\nSemantic Similarity : {cross_sim:.4f}')
    print(f'TF-IDF Similarity   : {tfidf_sim:.4f}')
    print(f'Difference          : {abs(cross_sim - tfidf_sim):.4f}')
    print(f'Common Keywords     : {common if common else "None"}')

    print('\n--- Groq AI Analysis ---')
    print(groq_compare_papers(r1, r2, cross_sim))
    print('='*70)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, title, kws, color in [
        (axes[0], r1['title'][:45], kw1, '#3498db'),
        (axes[1], r2['title'][:45], kw2, '#e74c3c')
    ]:
        ax.barh(range(len(kws)), [1]*len(kws), color=color, alpha=0.8)
        ax.set_yticks(range(len(kws)))
        ax.set_yticklabels(kws, fontsize=10)
        ax.set_title(title, fontsize=9, fontweight='bold')
        ax.set_xticks([])
    plt.suptitle(
        f'Keyword Comparison | Semantic Sim: {cross_sim:.3f} | TF-IDF Sim: {tfidf_sim:.3f}',
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('outputs/05_paper_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

compare_papers('BERT language model pre-training', 'GPT generative language model')

## PDF Export

In [ ]:
def export_pdf(query, results, summaries=None, keywords_list=None,
               filename='outputs/search_results.pdf'):
    os.makedirs('outputs', exist_ok=True)
    doc    = SimpleDocTemplate(filename, pagesize=letter,
                               rightMargin=0.75*inch, leftMargin=0.75*inch,
                               topMargin=1*inch, bottomMargin=0.75*inch)
    styles = getSampleStyleSheet()
    ts = ParagraphStyle('T', parent=styles['Title'], fontSize=16,
                         textColor=colors.HexColor('#1F4E79'))
    hs = ParagraphStyle('H', parent=styles['Heading2'], fontSize=11,
                         textColor=colors.HexColor('#2E75B6'))
    bs = ParagraphStyle('B', parent=styles['Normal'], fontSize=9, leading=13)
    ms = ParagraphStyle('M', parent=styles['Normal'], fontSize=8,
                         textColor=colors.gray)
    story = [
        Paragraph('AI Research Paper Intelligence System', ts),
        Paragraph('CBSOT Summer Internship 2026', ms),
        Spacer(1, 0.1*inch),
        HRFlowable(width='100%', thickness=2, color=colors.HexColor('#2E75B6')),
        Spacer(1, 0.1*inch),
        Paragraph(f'Query: <b>{query}</b>', bs),
        Paragraph(f'Papers: {len(results)} | {datetime.now().strftime("%d %B %Y")}', ms),
        Spacer(1, 0.15*inch),
    ]
    for i, r in enumerate(results):
        story += [Paragraph(f'{i+1}. {r["title"]}', hs),
                  Paragraph(f'Score: {r["score"]:.4f}', ms)]
        if summaries and i < len(summaries):
            story.append(Paragraph(f'<b>Summary:</b> {summaries[i]}', bs))
        if keywords_list and i < len(keywords_list):
            story.append(Paragraph(f'<b>Keywords:</b> {", ".join(keywords_list[i])}', bs))
        short = r['abstract'][:350]+'...' if len(r['abstract'])>350 else r['abstract']
        story += [Paragraph(f'<b>Abstract:</b> {short}', bs),
                  HRFlowable(width='100%', thickness=0.5, color=colors.lightgrey),
                  Spacer(1, 0.08*inch)]
    doc.build(story)
    print(f'PDF saved: {filename}')

query    = 'contrastive learning self-supervised'
results  = semantic_search(query, top_k=5)
sums     = [summarize(r['abstract']) for r in results]
kws_list = [extract_keywords(r['abstract']) for r in results]
export_pdf(query, results, sums, kws_list)

## Bookmarks & Search History

In [ ]:
BOOKMARKS_FILE = 'outputs/bookmarks.json'
HISTORY_FILE   = 'outputs/search_history.json'

def load_json(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return []

def save_json(path, data):
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

def add_bookmark(paper):
    bm = load_json(BOOKMARKS_FILE)
    if not any(b['title'] == paper['title'] for b in bm):
        bm.append({'title': paper['title'], 'abstract': paper['abstract'][:300],
                   'score': paper['score'], 'saved_at': datetime.now().isoformat()})
        save_json(BOOKMARKS_FILE, bm)
        print(f'Bookmarked: {paper["title"][:60]}')
    else:
        print('Already bookmarked!')

def add_to_history(query, num_results):
    h = load_json(HISTORY_FILE)
    h.insert(0, {'query': query, 'results': num_results,
                 'timestamp': datetime.now().isoformat()})
    save_json(HISTORY_FILE, h[:50])

def show_bookmarks():
    bm = load_json(BOOKMARKS_FILE)
    print(f'Bookmarks ({len(bm)}):')
    for i, b in enumerate(bm):
        print(f'  {i+1}. {b["title"][:60]}')

def show_history():
    h = load_json(HISTORY_FILE)
    print(f'History ({len(h)} searches):')
    for entry in h[:10]:
        print(f'  [{entry["timestamp"][:10]}] "{entry["query"]}" - {entry["results"]} results')

results = semantic_search('deep learning image classification', top_k=3)
add_to_history('deep learning image classification', len(results))
add_bookmark(results[0])
add_bookmark(results[1])

results2 = semantic_search('LSTM recurrent networks', top_k=3)
add_to_history('LSTM recurrent networks', len(results2))
add_bookmark(results2[0])

print()
show_bookmarks()
print()
show_history()